# Step 5: Automate model deployment with a SageMaker project

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.6.0</strong> and the <strong>SageMaker Python SDK version 3.27.0</strong>.
</div>

In the previous steps you built an ML pipeline that registers model versions in the SageMaker Model Registry, and you deployed and monitored an approved model version manually. In this notebook you automate the model deployment with a CI/CD pipeline.

You publish your own [SageMaker project template](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates-custom.html) and your own deployment seed code to Amazon S3, and create a SageMaker project from them. The project watches the model package group from step 3. When you approve a model version, the project deploys it to a staging endpoint, tests the endpoint, waits for your approval, and deploys the model to a production endpoint.

||||
|---|---|---|
|1. |Experiment in a notebook ||
|2. |Scale with SageMaker AI jobs and SageMaker SDK ||
|3. |Operationalize with ML pipeline, model registry ||
|4. |Deploy the model and add data and model monitoring ||
|5. |Automate model deployment with a SageMaker project |**<<<< YOU ARE HERE**|

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop **must** run on **Python 3.12**. The cell below will fail with an `AssertionError` if you are using a different Python version.

If you see an error, follow these steps:

1. Open a **Terminal** in JupyterLab (**File → New → Terminal**)
2. Run the setup script:
   ```bash
   cd ~/amazon-sagemaker-from-idea-to-production
   ./setup-py312.sh
   ```
3. **Refresh your browser** to pick up the new kernel
4. In this notebook, go to **Kernel → Change Kernel → Python 3.12**
5. **Restart the kernel** and re-run from the top

</div>

## What you do in this notebook

1. Explore and package the deployment seed code in the [`deployment-seed-code`](deployment-seed-code/) folder.
2. Publish the seed code and the project template [`sm-custom-projects/deployment-only-project-template.yaml`](sm-custom-projects/deployment-only-project-template.yaml) to your S3 bucket.
3. Create a SageMaker project from the template.
4. Follow the deployment pipeline: staging deployment, endpoint test, your approval, production deployment.

You don't need a GitHub account or a code connection. The project creates its own AWS CodeCommit repository and fills it with the seed code from S3.

<div class="alert alert-info"><b>Cost:</b> the project creates CodeCommit, CodeBuild, CodePipeline, EventBridge, CloudFormation, and S3 resources. Each pipeline run creates or updates a staging endpoint and, after your approval, a production endpoint. Run the <a href="99-clean-up.ipynb">clean-up notebook</a> when you're done.</div>

## Architecture

![SageMaker project deployment toolchain with Model Registry and source triggers, staged deployment, testing, manual approval, and production](img/deployment-project-architecture-aws.svg)

The project's CloudFormation stack owns the CodeCommit repository, the EventBridge rules, the artifact bucket, the CodeBuild projects, and the CodePipeline pipeline. Each deployment stage is a separate CloudFormation stack with a SageMaker model, an endpoint configuration with data capture, and an endpoint.

Two EventBridge rules start the pipeline:
- a model package version in the model package group is approved or rejected
- a commit is pushed to the `main` branch of the project's CodeCommit repository

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

In [ ]:
import boto3
import hashlib
import json
import zipfile
from pathlib import Path
from time import gmtime, strftime, sleep

import numpy as np
import yaml
from botocore.exceptions import ClientError
from IPython.display import HTML

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
sm = boto3.client("sagemaker")
s3 = boto3.client("s3")
cfn = boto3.client("cloudformation")
iam = boto3.client("iam")
codepipeline = boto3.client("codepipeline")

In [ ]:
try:
    print(f"Model package group: {target_model_package_group_name}")
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++")
    print("You must complete the notebook 03-sagemaker-pipeline   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++")

---

## Explore the deployment seed code

The seed code is the initial content of the project's code repository. It's a regular folder in this workshop repository: [`deployment-seed-code`](deployment-seed-code/). You own it, version it, and review it like any other code. When the project is created, it copies the seed code into a new CodeCommit repository. From then on, the project's repository is the source of the deployment pipeline.

The seed code is based on the deployment seed code of the [SageMaker-provided MLOps project templates](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates-sm.html). The file [`ORIGIN.md`](deployment-seed-code/ORIGIN.md) lists the changes.

|File|Purpose|
|---|---|
|`buildspec.yml`|CodeBuild instructions for the `Build` stage. Runs `build.py` and packages the endpoint CloudFormation template.|
|`build.py`|Selects the newest `Approved` model package version in the model package group and writes the CloudFormation parameter files for staging and production.|
|`endpoint-config-template.yml`|CloudFormation template with the SageMaker model, endpoint configuration with data capture, and endpoint.|
|`staging-config.json`, `prod-config.json`|Stage settings: instance type and count, data capture sampling percentage.|
|`test/buildspec.yml`, `test/test.py`|CodeBuild instructions and the test script for the `TestStaging` action. The test sends a real inference request to the staging endpoint.|
|`test/payload.csv`|One held-out record with the 64 features the model expects.|

Open the files in the JupyterLab file browser to see the details. The next cell shows the stage settings.

In [ ]:
seed_directory = Path("deployment-seed-code")

for config_file in ["staging-config.json", "prod-config.json"]:
    print(f"{config_file}:")
    print(json.dumps(json.loads((seed_directory / config_file).read_text()), indent=2))

### Package the seed code
CodeCommit expands the root of the ZIP archive into the root of the new repository. The next cell packages the seed folder content at the archive root and skips local artifacts such as `.ipynb_checkpoints`.

In [ ]:
build_directory = Path("tmp")
build_directory.mkdir(exist_ok=True)
seed_zip = build_directory / "deployment-seed-code.zip"

seed_files = sorted(
    p for p in seed_directory.rglob("*")
    if p.is_file()
    and not {"__pycache__", ".ipynb_checkpoints"}.intersection(p.parts)
    and p.suffix != ".pyc"
    and p.name != "ORIGIN.md"  # documents the seed code in this workshop repository only
)

# Fixed timestamps make the archive reproducible: the same seed code gives the same ZIP file
with zipfile.ZipFile(seed_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for source_path in seed_files:
        info = zipfile.ZipInfo(source_path.relative_to(seed_directory).as_posix(), date_time=(2024, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        archive.writestr(info, source_path.read_bytes())

with zipfile.ZipFile(seed_zip) as archive:
    print("\n".join(archive.namelist()))

seed_sha256 = hashlib.sha256(seed_zip.read_bytes()).hexdigest()
print(f"\nBuilt {seed_zip} ({seed_zip.stat().st_size:,} bytes), SHA-256: {seed_sha256}")

---

## Explore the project template
The project template [`sm-custom-projects/deployment-only-project-template.yaml`](sm-custom-projects/deployment-only-project-template.yaml) is a CloudFormation template. SageMaker sets the parameters `SageMakerProjectName` and `SageMakerProjectId` when it creates a project. You provide the other parameters:

- `SourceModelPackageGroupName`: the model package group whose approved versions the project deploys
- `DeploymentSeedCodeBucket` and `DeploymentSeedCodeKey`: the S3 location of the seed code ZIP file

The template creates the following resources:
- a CodeCommit repository with the seed code
- EventBridge rules for model package state changes and for commits to the `main` branch
- a CodeBuild project to build the endpoint CloudFormation template, and a CodeBuild project to test the staging endpoint
- a CodePipeline pipeline with the stages `Source`, `Build`, `DeployStaging` (deploy, test, manual approval), and `DeployProd`
- an S3 bucket `sagemaker-project-<project-id>` for the pipeline artifacts and the endpoint data capture

The next cell lists the resources and the pipeline stages.

In [ ]:
project_template_path = Path("sm-custom-projects/deployment-only-project-template.yaml")

# CloudFormation short-form functions like !Ref and !Sub are not standard YAML tags, so ignore them for this overview
class CfnLoader(yaml.SafeLoader):
    pass

CfnLoader.add_multi_constructor("!", lambda loader, suffix, node: None)

project_template = yaml.load(project_template_path.read_text(), Loader=CfnLoader)

print(f"Parameters: {', '.join(project_template['Parameters'])}\n")
for logical_id, resource in project_template["Resources"].items():
    print(f"{resource['Type']:<32} {logical_id}")

stages = [s["Name"] for s in project_template["Resources"]["ModelDeployPipeline"]["Properties"]["Stages"]]
print(f"\nPipeline stages: {' -> '.join(stages)}")

---

## IAM roles for SageMaker projects

SageMaker projects use two kinds of IAM roles.

**Launch role**: `AmazonSageMakerProjectsLaunchRole` has the permissions to create the project's resources, such as the CodeCommit repository, CodeBuild projects, the CodePipeline pipeline, and the S3 bucket. SageMaker uses this role when you create a project. Data scientists and ML engineers don't need these broad permissions themselves. They only need permission to pass the launch role.

**Use roles**: each AWS service in the project uses its own role with only the permissions it needs:
- `AmazonSageMakerProjectsEventsRole`: EventBridge starts the pipeline
- `AmazonSageMakerProjectsCodePipelineRole`: CodePipeline orchestrates the stages
- `AmazonSageMakerProjectsCodeBuildRole`: CodeBuild builds the deployment templates and tests the staging endpoint
- `AmazonSageMakerProjectsCloudformationRole`: CloudFormation creates the endpoints
- `AmazonSageMakerProjectsExecutionRole`: the SageMaker model uses this role to pull the container image and the model artifacts

The CloudFormation template [`sm-custom-projects/sagemaker-projects-roles-and-policies.yaml`](sm-custom-projects/sagemaker-projects-roles-and-policies.yaml) defines all roles. The launch role includes the CodeCommit permissions to create the seeded repository.

<div class="alert alert-info">In an AWS-provided workshop account the roles already exist. If you provisioned the roles yourself with an older version of the template, the next cell updates the stack to add the CodeCommit permissions.</div>

In [ ]:
roles_stack_name = "sagemaker-projects-roles-policies"
roles_template_body = Path("sm-custom-projects/sagemaker-projects-roles-and-policies.yaml").read_text()
launch_role_name = "AmazonSageMakerProjectsLaunchRole"

def roles_stack_exists():
    try:
        return cfn.describe_stacks(StackName=roles_stack_name)["Stacks"][0]["StackStatus"] != "DELETE_COMPLETE"
    except ClientError:
        return False

try:
    iam.get_role(RoleName=launch_role_name)
    role_exists = True
except iam.exceptions.NoSuchEntityException:
    role_exists = False

stack_args = dict(
    StackName=roles_stack_name,
    TemplateBody=roles_template_body,
    Capabilities=["CAPABILITY_NAMED_IAM"],
    Parameters=[{"ParameterKey": "SageMakerExecutionRoleArn", "ParameterValue": sm_role}],
)

if not role_exists:
    print(f"{launch_role_name} doesn't exist, create the stack {roles_stack_name}")
    cfn.create_stack(**stack_args)
    cfn.get_waiter("stack_create_complete").wait(StackName=roles_stack_name)
elif roles_stack_exists():
    try:
        cfn.update_stack(**stack_args)
        print(f"Updating the stack {roles_stack_name}")
        cfn.get_waiter("stack_update_complete").wait(StackName=roles_stack_name)
    except ClientError as e:
        if "No updates are to be performed" not in str(e):
            raise
        print(f"The stack {roles_stack_name} is up to date")
else:
    print(f"{launch_role_name} exists and was provisioned outside of this notebook, use it as is")

launch_role_arn = iam.get_role(RoleName=launch_role_name)["Role"]["Arn"]
print(f"Launch role: {launch_role_arn}")

---

## Publish the template and the seed code to S3
SageMaker creates projects from templates stored in S3. The next cell uploads the seed code ZIP file and the project template to your S3 bucket. The template object gets the tag `sagemaker:studio-visibility=true`, so that Studio shows it in the list of organization templates.

In [ ]:
template_key = "templates/deployment-only-project-template.yaml"
seed_code_key = "project-seed-code/deployment-seed-code.zip"

s3.upload_file(str(seed_zip), bucket_name, seed_code_key, ExtraArgs={"Metadata": {"sha256": seed_sha256}})
s3.upload_file(str(project_template_path), bucket_name, template_key)
s3.put_object_tagging(
    Bucket=bucket_name,
    Key=template_key,
    Tagging={"TagSet": [{"Key": "sagemaker:studio-visibility", "Value": "true"}]},
)

template_url = f"https://{bucket_name}.s3.{region}.amazonaws.com/{template_key}"
print(f"Seed code: s3://{bucket_name}/{seed_code_key}")
print(f"Template:  {template_url}")

### Show the template in Studio
To list S3 templates in the Studio UI, the SageMaker domain needs the tag `sagemaker:projectS3TemplatesLocation` with the S3 location of the templates, and the bucket needs a CORS policy that allows Studio to read the templates. You need this only if you want to create the project in the Studio UI (Option 2 below).

In [ ]:
domain_arn = sm.describe_domain(DomainId=domain_id)["DomainArn"]
sm.add_tags(
    ResourceArn=domain_arn,
    Tags=[{"Key": "sagemaker:projectS3TemplatesLocation", "Value": f"s3://{bucket_name}/templates/"}],
)

with open("sm-custom-projects/cors-policy.json") as f:
    s3.put_bucket_cors(Bucket=bucket_name, CORSConfiguration=json.load(f))

print(f"Studio lists templates from s3://{bucket_name}/templates/")

---

## Make sure there is an approved model version
The `Build` stage deploys the newest model version with the status `Approved`. CodePipeline runs the pipeline once right after the project creates it, so an approved version must exist before you create the project. Otherwise the first run fails in the `Build` stage.

If you completed step 4, you already approved a model version. If not, the next cell approves the newest version with the status `PendingManualApproval`.

In [ ]:
def list_model_packages(status):
    return [
        p for page in sm.get_paginator("list_model_packages").paginate(
            ModelPackageGroupName=target_model_package_group_name,
            ModelApprovalStatus=status,
            SortBy="CreationTime",
            SortOrder="Descending",
        )
        for p in page["ModelPackageSummaryList"]
    ]

def is_deployable(model_package_arn):
    return "InferenceSpecification" in sm.describe_model_package(ModelPackageName=model_package_arn)

approved_packages = [p for p in list_model_packages("Approved") if is_deployable(p["ModelPackageArn"])]

if approved_packages:
    print(f"Approved model version to deploy: {approved_packages[0]['ModelPackageArn']}")
else:
    pending_packages = [p for p in list_model_packages("PendingManualApproval") if is_deployable(p["ModelPackageArn"])]
    if not pending_packages:
        raise Exception(f"No deployable model version in {target_model_package_group_name}. Run the pipeline in notebook 03 first.")
    sm.update_model_package(ModelPackageArn=pending_packages[0]["ModelPackageArn"], ModelApprovalStatus="Approved")
    print(f"Approved model version: {pending_packages[0]['ModelPackageArn']}")

Click on the link constructed by the next cell to see the model versions in the Studio UI:

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/registry/registered-models/{}/versions">the model package versions</a> in the Studio UI</b>'.format(
            domain_id, region, target_model_package_group_name))
)

---

## Create a SageMaker project
You can create a project programmatically in this notebook - **Option 1** - or in the Studio UI - **Option 2**.

Option 1 is recommended as it has no dependency on the UI.<br/>
Option 2 demonstrates the [**Create project** UI flow](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-create.html).

### Option 1: Create a project programmatically - recommended
The `CreateProject` API with a `CfnTemplateProvider` reads the template directly from S3. You don't need AWS Service Catalog.

The next cell reuses the project if you already created it in a previous run of this notebook.

In [ ]:
project_name = None
try:
    if sm.describe_project(ProjectName=deployment_project_name)["ProjectStatus"] in ["CreateInProgress", "CreateCompleted"]:
        project_name = deployment_project_name
        print(f"Reuse the existing project {project_name}")
except (NameError, ClientError):
    pass

if project_name is None:
    project_name = f"deploy-{strftime('%d-%H-%M-%S', gmtime())}"

    r = sm.create_project(
        ProjectName=project_name,
        ProjectDescription="Deployment of approved model versions to staging and production endpoints",
        TemplateProviders=[
            {
                "CfnTemplateProvider": {
                    "TemplateName": "DeploymentOnlyProject",
                    "TemplateURL": template_url,
                    "RoleARN": launch_role_arn,
                    "Parameters": [
                        {"Key": "SourceModelPackageGroupName", "Value": target_model_package_group_name},
                        {"Key": "DeploymentSeedCodeBucket", "Value": bucket_name},
                        {"Key": "DeploymentSeedCodeKey", "Value": seed_code_key},
                    ],
                }
            }
        ],
    )
    print(f"Creating the project {r['ProjectArn']}")

<div class="alert alert-info">Wait until the project creation is completed by running the next cell. It takes 1-2 minutes.</div>

In [ ]:
status = sm.describe_project(ProjectName=project_name)["ProjectStatus"]
while status not in ["CreateCompleted", "CreateFailed"]:
    print("Waiting for project creation completion")
    sleep(10)
    status = sm.describe_project(ProjectName=project_name)["ProjectStatus"]

print(f"Project {project_name} status: {status}")

In [ ]:
project_data = sm.describe_project(ProjectName=project_name)
if project_data["ProjectStatus"] != "CreateCompleted":
    # The most common cause is a launch role without the CodeCommit permissions from sm-custom-projects/sagemaker-projects-roles-and-policies.yaml
    print(project_data.get("ProjectStatusMessage", ""))
assert project_data["ProjectStatus"] == "CreateCompleted", "Project status must be CreateCompleted!"

### End of Option 1: Create a project programmatically
Navigate to the section **Explore the project**.

---

### Option 2: Create a project in the Studio UI
<div class="alert alert-info"><b>Skip this section if you have already created a project programmatically</b></div>

1. In Studio, choose **Deployments** > **Projects** > **Create project**
2. Choose **Organization templates** > **S3 Templates**

![](img/sm-projects-custom-template-s3.png)

3. Select the template `deployment-only-project-template` and choose **Next**
4. Enter a project name and the template parameters printed by the next cell. In the **Role ARN** field, enter the ARN of the launch role. This field is optional, but without it the SageMaker execution role of your user profile needs all the permissions to create the project resources.
5. Choose **Create** and wait until the project is created

In [ ]:
print(f"Model package group name: {target_model_package_group_name}")
print(f"Seed code S3 bucket:      {bucket_name}")
print(f"Seed code S3 key:         {seed_code_key}")
print(f"Role ARN:                 {launch_role_arn}")

Set the variable `project_name` in the next cell to the name of the project you created in the Studio UI and uncomment the line.

In [ ]:
# project_name = "<SET TO THE NAME OF THE CREATED PROJECT>" # Keep commented out if you used option 1 to create a project

### End of Option 2: Create a project in the Studio UI

---

## Explore the project
The project names all resources after the project name and the project ID. The next cell stores the project name and ID for the clean-up notebook and the additional topics notebooks.

In [ ]:
project_data = sm.describe_project(ProjectName=project_name)
assert project_data["ProjectStatus"] == "CreateCompleted", "Project status must be CreateCompleted!"
project_id = project_data["ProjectId"]
deployment_project_name = project_name

pipeline_name = f"sagemaker-{project_name}-{project_id}-modeldeploy"
repository_name = f"sagemaker-{project_name}-{project_id}-modeldeploy"
staging_endpoint_name = f"{project_name}-staging"
prod_endpoint_name = f"{project_name}-prod"

print(f"Project:           {project_name} ({project_id})")
print(f"Pipeline:          {pipeline_name}")
print(f"Repository:        {repository_name}")
print(f"Staging endpoint:  {staging_endpoint_name}")
print(f"Prod endpoint:     {prod_endpoint_name}")

%store project_name
%store project_id
%store deployment_project_name

Click on the links constructed by the next cell to see the project in the Studio UI, the deployment pipeline in the CodePipeline console, and the seed code in the CodeCommit console.

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/projects/{}/">the project</a> in the Studio UI</b>'.format(
            domain_id, region, project_name))
)
display(
    HTML('<b>See <a target="top" href="https://console.aws.amazon.com/codesuite/codepipeline/pipelines/{}/view?region={}">the deployment pipeline</a> in the CodePipeline console</b>'.format(
            pipeline_name, region))
)
display(
    HTML('<b>See <a target="top" href="https://console.aws.amazon.com/codesuite/codecommit/repositories/{}/browse?region={}">the deployment code repository</a> in the CodeCommit console</b>'.format(
            repository_name, region))
)

---

## Deployment pipeline execution
The deployment pipeline performs the following actions:
1. `Source`: get the code from the `main` branch of the CodeCommit repository
1. `Build`: select the newest approved model version and create the CloudFormation parameter files for staging and production
1. `DeployStaging`: create a SageMaker real-time inference endpoint with the name `<PROJECT-NAME>-staging`, run the test script on the staging endpoint, and wait for manual approval
1. `DeployProd`: create a SageMaker endpoint with the name `<PROJECT-NAME>-prod`

The following image shows a pipeline execution that waits for the manual approval:

![CodePipeline execution waiting for production approval](img/project-codepipeline.png)

CodePipeline starts the first execution right after the project creates the pipeline. The next cell waits for this execution and starts one if there is none.

In [ ]:
def latest_pipeline_execution():
    executions = codepipeline.list_pipeline_executions(pipelineName=pipeline_name, maxResults=1)["pipelineExecutionSummaries"]
    return executions[0] if executions else None

for _ in range(12):
    execution = latest_pipeline_execution()
    if execution:
        break
    sleep(10)
else:
    codepipeline.start_pipeline_execution(name=pipeline_name)
    sleep(5)
    execution = latest_pipeline_execution()

print(f"Pipeline execution {execution['pipelineExecutionId']}: {execution['status']}")

The staging deployment takes about 5-10 minutes. The next cell waits until the staging endpoint is `InService`. If a pipeline action fails, the cell stops and shows the error with a link to the logs.

In [ ]:
def pipeline_failure():
    """Return a description of the first failed action of the latest execution or None."""
    execution = latest_pipeline_execution()
    if execution["status"] not in ["Failed", "Stopped", "Superseded"]:
        return None
    for stage in codepipeline.get_pipeline_state(name=pipeline_name)["stageStates"]:
        for action in stage["actionStates"]:
            latest = action.get("latestExecution", {})
            if latest.get("status") == "Failed":
                message = latest.get("errorDetails", {}).get("message", "")
                return f"{stage['stageName']}/{action['actionName']} failed: {message}\nLogs: {latest.get('externalExecutionUrl', '')}"
    return f"Pipeline execution is {execution['status']}"

def wait_for_endpoint(endpoint_name):
    while True:
        try:
            status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        except ClientError:
            status = "NotCreated"
        if status == "InService":
            break
        if status == "Failed":
            raise Exception(sm.describe_endpoint(EndpointName=endpoint_name).get("FailureReason"))
        failure = pipeline_failure()
        if failure:
            raise Exception(failure)
        print(f"{strftime('%H:%M:%S', gmtime())} Endpoint {endpoint_name}: {status}")
        sleep(30)
    print(f"Endpoint {endpoint_name} is InService")

wait_for_endpoint(staging_endpoint_name)

You can see the endpoint in the Studio UI in **Deployments** > **Endpoints**:

![](img/staging-endpoint.png)

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/inference-experience/endpoints">inference endpoints</a> in the Studio UI</b>'.format(
            domain_id, region))
)

### Test the staging endpoint
The `TestStaging` action in the pipeline sends one record to the staging endpoint. Send the full test dataset from step 2 and calculate the ROC AUC to see the model performance on the staging endpoint.

In [ ]:
from sklearn.metrics import roc_auc_score

sm_runtime = boto3.client("sagemaker-runtime")

## Make sure you have executed Notebook 02-sagemaker-containers.ipynb
test_x = np.loadtxt("tmp/test_x.csv", delimiter=",")
test_y = np.loadtxt("tmp/test_y.csv", delimiter=",")

def predict(endpoint_name, x):
    response = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="text/csv",
        Accept="text/csv",
        Body="\n".join(",".join(map(str, row)) for row in x),
    )
    return np.array(response["Body"].read().decode("utf-8").strip().split("\n"), dtype=float)

predictions = predict(staging_endpoint_name, test_x)
print(f"Staging endpoint ROC AUC on {len(test_y)} test records: {roc_auc_score(test_y, predictions):.4f}")

---

## Deploy the model version to production
After the `TestStaging` action succeeds, the pipeline waits in the `ApproveDeployment` action. Approving this action deploys the model to the production endpoint. You can approve in the CodePipeline console or programmatically.

<div class="alert alert-info">The production endpoint is a second billable endpoint.</div>

### Option 1: Approve in the CodePipeline console
Click on the link constructed by the next cell. In the **DeployStaging** stage, choose **Review** on the **ApproveDeployment** action. Wait until the `TestStaging` action has the status `Succeeded`.

<img src="img/deploy-staging-review.png" alt="approve deployment" width="800px">

In the **Review** dialog box, select **Approve** and choose **Submit**.

In [ ]:
display(
    HTML('<b>Approve the manual step in <a target="top" href="https://console.aws.amazon.com/codesuite/codepipeline/pipelines/{}/view?region={}">AWS CodePipeline</a></b>'.format(
            pipeline_name, region))
)

### Option 2: Approve programmatically
The next cell waits until the approval action is active, and then approves it. If you already approved in the console, the cell does nothing.

In [ ]:
def approval_action_state():
    stage = next(s for s in codepipeline.get_pipeline_state(name=pipeline_name)["stageStates"] if s["stageName"] == "DeployStaging")
    action = next(a for a in stage["actionStates"] if a["actionName"] == "ApproveDeployment")
    return action.get("latestExecution", {})

while True:
    approval = approval_action_state()
    if approval.get("status") in ["InProgress", "Succeeded"]:
        break
    failure = pipeline_failure()
    if failure:
        raise Exception(failure)
    print(f"{strftime('%H:%M:%S', gmtime())} Waiting for the ApproveDeployment action")
    sleep(20)

if approval["status"] == "Succeeded":
    print("The production deployment is already approved")
else:
    codepipeline.put_approval_result(
        pipelineName=pipeline_name,
        stageName="DeployStaging",
        actionName="ApproveDeployment",
        result={"summary": "Staging endpoint tested in notebook 05", "status": "Approved"},
        token=approval["token"],
    )
    print("Approved the production deployment")

The production deployment takes about 5-10 minutes. The next cell waits until the production endpoint is `InService`.

In [ ]:
wait_for_endpoint(prod_endpoint_name)

predictions = predict(prod_endpoint_name, test_x)
print(f"Production endpoint ROC AUC on {len(test_y)} test records: {roc_auc_score(test_y, predictions):.4f}")

Click on the link constructed by the next cell to see the endpoints of the project. Both endpoints, `staging` and `prod`, are visible in the project because the deployment pipeline tags them with the project name and ID.

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/projects/{}/endpoints">endpoints of the project {}</a> in the Studio UI</b>'.format(
            domain_id, region, project_name, project_name))
)

---

## Optional: change the deployment with a commit
The project's CodeCommit repository is the source of the deployment pipeline. Every push to the `main` branch starts the pipeline. For example, you can change the instance type in `prod-config.json` or add autoscaling to `endpoint-config-template.yml`.

To work with the repository:
1. Open a new terminal window via **File** > **New** > **Terminal**
2. Copy the output of the next cell and paste it into the terminal command line. The commands configure git to use your IAM credentials for CodeCommit and clone the repository into your home directory.
3. Change a file, for example set `"SamplingPercentage": "50"` in `prod-config.json`, and run `git commit -am "change sampling" && git push`

<div class="alert alert-info">A push starts a new pipeline execution that updates the staging endpoint and waits for your approval before it updates the production endpoint.</div>

The SageMaker execution role of your user profile needs `codecommit:GitPull` and `codecommit:GitPush` permissions for the repository. The managed policy `AmazonSageMakerFullAccess` grants these for repositories with `sagemaker` in the name.

Changes in the project repository affect only this project. To change the seed code for new projects, edit the `deployment-seed-code` folder and publish it again.

In [ ]:
cmd = f'''git config --global credential.helper '!aws codecommit credential-helper $@'
git config --global credential.UseHttpPath true

cd ~
git clone https://git-codecommit.{region}.amazonaws.com/v1/repos/{repository_name}
cd {repository_name}'''

display(HTML(f'<pre>{cmd}</pre>'))

---

## Summary
In this notebook you implemented an automated CI/CD deployment pipeline with the following features:
- you own the project template and the seed code, and publish them to S3 without a code connection to an external Git provider
- a model approval in the model registry starts the deployment pipeline
- a commit to the project's CodeCommit repository starts the deployment pipeline
- CloudFormation templates deploy the SageMaker real-time inference endpoints as infrastructure as code
- the pipeline deploys to staging, tests the staging endpoint with a real inference request, and deploys to production after a manual approval

## Clean-up
<div style="border: 4px solid coral; text-align: center; margin: auto;">
    <p style=" text-align: center; margin: auto;">
    If you're going to run the optional <a href="additional-topics/04-monitoring-sagemaker-native.ipynb">SageMaker Model Monitor notebook</a>, keep at least one of the endpoints. Otherwise navigate to the <b>clean-up notebook (99-clean-up.ipynb)</b> and follow the clean-up instructions to avoid charges in your AWS account.
    <br>
    <br>
    You don't need to run the clean-up if you're using an AWS-provided AWS account.
    </p>
</div>

## Finish the workshop
This is the last step of the workshop. To avoid charges in your AWS account, run the [clean-up notebook](99-clean-up.ipynb).

## Further development ideas for your real-world projects
- Add a CI/CD pipeline for model building. The notebooks [additional-topics/05-sagemaker-project](additional-topics/05-sagemaker-project.ipynb) and [additional-topics/06-deploy](additional-topics/06-deploy.ipynb) show a project with both model building and model deployment pipelines connected to GitHub repositories
- Add [multi-account model deployment](https://aws.amazon.com/blogs/machine-learning/multi-account-model-deployment-with-amazon-sagemaker-pipelines/) to your ML workflow
- Extend `build.py` to also check the model lifecycle stage or model metrics before selecting a model version
- Add autoscaling, alarms, or [deployment guardrails](https://docs.aws.amazon.com/sagemaker/latest/dg/deployment-guardrails.html) to `endpoint-config-template.yml`
- Add end-to-end data encryption using AWS KMS keys
- Use [Amazon SageMaker Inference Recommender](https://docs.aws.amazon.com/sagemaker/latest/dg/inference-recommender.html) to run automated load tests for your inference endpoints and to select the best instance type and configuration

## Additional resources
- [Create custom project templates](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates-custom.html)
- [Deploy a Machine Learning Model to a Real-Time Inference Endpoint](https://aws.amazon.com/getting-started/hands-on/machine-learning-tutorial-deploy-model-to-real-time-inference-endpoint/)
- [Amazon SageMaker secure MLOps](https://github.com/aws-samples/amazon-sagemaker-secure-mlops)
- [Testing approaches for Amazon SageMaker ML models](https://aws.amazon.com/blogs/machine-learning/testing-approaches-for-amazon-sagemaker-ml-models/)
- [Model hosting patterns in Amazon SageMaker blog series](https://aws.amazon.com/blogs/machine-learning/model-hosting-patterns-in-amazon-sagemaker-part-1-common-design-patterns-for-building-ml-applications-on-amazon-sagemaker/)
- [Take advantage of advanced deployment strategies using Amazon SageMaker deployment guardrails](https://aws.amazon.com/blogs/machine-learning/take-advantage-of-advanced-deployment-strategies-using-amazon-sagemaker-deployment-guardrails/)
- [MLOps deployment best practices for real-time inference model serving endpoints with Amazon SageMaker](https://aws.amazon.com/blogs/machine-learning/mlops-deployment-best-practices-for-real-time-inference-model-serving-endpoints-with-amazon-sagemaker/)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>